# e5 · Linear vs non-linear, Elastic Net vs XGBoost

Does non-linearity add anything? Each pair holds the information set fixed and compares **XGBoost (a)** with **Elastic Net (b)**:

| Pair | Information set | a (XGBoost) | b (Elastic Net) |
|---|---|---|---|
| E5.1 | Full macro + VIX | `xgb_full_macro` | `en_full_macro` |
| E5.2 | VIX only | `xgb_vix_only` | `en_vix_only` |
| E5.3 | Macro without VIX | `xgb_no_vix` | `en_no_vix` |

**Tests.** EN and XGBoost are different model classes fed the same inputs, so neither nests the other. Diebold-Mariano (DM) is therefore valid for **both** losses, and Clark-West is not needed here (professor feedback). As in e3, the loss differential is averaged across ETFs each month and tested with Newey-West (Andrews bandwidth). p-values are one-sided, H1: *a beats b*, i.e. **XGBoost beats Elastic Net**.

**Output** in `forecasting/results/`: `e5_losses.csv`, `e5_evaluation.csv`. `gain_t > 0` favours a (XGBoost). `loss_ratio` is a's loss over b's, i.e. XGBoost / EN, so a ratio below 1 means XGBoost has the lower average loss.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

MODELS_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research_config.py").exists())
sys.path.insert(0, str(MODELS_DIR))
from research_config import TICKERS

FORECAST_DIR = MODELS_DIR / "forecasting" / "forecasts"
EN_PATH      = FORECAST_DIR / "en_forecasts.csv"
XGB_PATH     = FORECAST_DIR / "xgb_forecasts_expanding_tidy.csv"
RESULTS_DIR  = MODELS_DIR / "forecasting" / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Pairs are (pair_id, a, b, question). a = XGBoost, b = Elastic Net, so "a beats b" = non-linearity helps.
# Defined here because I do not know whether research_config.BLOCKS has an entry for e5.
block = {
    "models": ["en_full_macro", "xgb_full_macro", "en_vix_only", "xgb_vix_only", "en_no_vix", "xgb_no_vix"],
    "pairs": [
        ("E5.1", "xgb_full_macro", "en_full_macro", "Full macro + VIX"),
        ("E5.2", "xgb_vix_only",   "en_vix_only",   "VIX only"),
        ("E5.3", "xgb_no_vix",     "en_no_vix",     "Macro without VIX"),
    ],
}

COLS = ["model", "etf", "date", "actual_logRV", "actual_RV", "fc_log", "fc_level"]
df = pd.concat([pd.read_csv(EN_PATH, parse_dates=["date"])[COLS],
                pd.read_csv(XGB_PATH, parse_dates=["date"])[COLS]], ignore_index=True)
available = set(df["model"])
pairs = [p for p in block["pairs"] if p[1] in available and p[2] in available]
print(f"{len(pairs)} of {len(block['pairs'])} E5 pairs runnable")

### Test helpers

Same helpers as e3. `hac_t` is the t-statistic of the mean loss differential with a Newey-West variance and an automatic (Andrews) bandwidth. This is the DM statistic: the mean of $d_t = L^{a}_t - L^{b}_t$ divided by its HAC standard error. `qlike` is the normalised QLIKE loss, $RV/F - \ln(RV/F) - 1$.

In [ ]:
def andrews_bandwidth(x):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))

def newey_west_lrv(x, lag):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    lag = max(0, min(lag, n - 1)); s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)

def hac_t(d):
    '''t-statistic of mean(d) with a Newey-West variance.'''
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))

def qlike(rv, fc):
    r = rv / fc
    return r - np.log(r) - 1

## Losses

Common sample: the (ETF, month) rows where all six models have a forecast. The XGBoost file has one extra early month (January 2008) that EN does not, and the common-sample filter drops it, so every model is scored on identical ETF-months. The check below confirms the two files agree on the realised values.

In [ ]:
models = [m for m in block["models"] if m in available]
d = df[df["model"].isin(models)].copy()
d = d[d.groupby(["etf", "date"])["model"].transform("nunique") == len(models)]

# the EN and XGB files must record identical realised values for the same (ETF, month)
spread = d.groupby(["etf", "date"])[["actual_logRV", "actual_RV"]].agg(lambda x: x.max() - x.min())
assert spread.max().max() < 1e-9, "EN/XGB realised values differ"

d["MSE_log"]   = (d["actual_logRV"] - d["fc_log"]) ** 2
d["MSE_level"] = (d["actual_RV"] - d["fc_level"]) ** 2
d["QLIKE"]     = qlike(d["actual_RV"], d["fc_level"])
print(f"{d['date'].min():%Y-%m} to {d['date'].max():%Y-%m}, {len(d)} rows")

LOSSES = ["QLIKE", "MSE_log", "MSE_level"]
per_etf = d.groupby(["model", "etf"])[LOSSES].mean().reset_index()
pooled = d.groupby("model")[LOSSES].mean().reset_index().assign(etf="pooled")
losses = pd.concat([per_etf, pooled], ignore_index=True).rename(columns={"etf": "ETF"})
losses.to_csv(RESULTS_DIR / "e5_losses.csv", index=False)
losses[losses["ETF"] == "pooled"].set_index("model")[LOSSES].round(4)

## Tests

For each pair, the monthly loss differential is $d_t = L^{XGB}_t - L^{EN}_t$ (negative favours XGBoost). The test statistic is `gain_t = -hac_t(d)`, so **`gain_t > 0` favours XGBoost**. The pooled row averages $d_t$ across the five ETFs each month before testing. The ETF rows test each ETF's own series. The p-value is one-sided, $H_1$: XGBoost beats EN.

Both losses use DM, because the pairs are non-nested. A significant result means XGBoost is better. An insignificant result means the two cannot be told apart, not that they are equally good.

In [ ]:
def one_sided_p(t):
    return float(1 - norm.cdf(t))

loss_idx = losses.set_index(["model", "ETF"])
rows = []
for pair_id, a, b, _ in pairs:
    A = d[d["model"] == a].set_index(["date", "etf"])
    B = d[d["model"] == b].set_index(["date", "etf"])
    diffs = {"QLIKE": A["QLIKE"] - B["QLIKE"],        # negative favours a
             "MSE_log": A["MSE_log"] - B["MSE_log"]}  # negative favours a
    for etf in ["pooled"] + TICKERS:
        for loss, diff in diffs.items():
            series = diff.groupby("date").mean() if etf == "pooled" else diff.xs(etf, level="etf")
            t = -hac_t(series)
            rows.append({
                "pair": pair_id, "a": a, "b": b, "ETF": etf, "loss": loss, "test": "DM",
                "loss_ratio": loss_idx.loc[(a, etf), loss] / loss_idx.loc[(b, etf), loss],
                "gain_t": t, "p_value": one_sided_p(t),
            })

tests = pd.DataFrame(rows)
tests.to_csv(RESULTS_DIR / "e5_evaluation.csv", index=False)
tests[tests["ETF"] == "pooled"].drop(columns="ETF").round(4).reset_index(drop=True)

## By sector (exploratory)

Each cell is `loss_ratio (p-value)`; a ratio below 1 means XGBoost (a) is better. Five sectors times three pairs with no multiple-testing correction, so a single sector's p-value is weak evidence; the pooled column is the formal result. Both losses use Diebold-Mariano.

In [ ]:
SECTORS = {"XLB": "Materials", "XLE": "Energy", "XLF": "Financials", "XLI": "Industrials", "XLY": "Cons. Disc."}

def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def by_sector(loss):
    t = tests[tests["loss"] == loss].copy()
    t["cell"] = [f"{r:.2f} ({fmt_p(p)})" for r, p in zip(t["loss_ratio"], t["p_value"])]
    t["row"] = t["pair"] + ": " + t["a"] + " vs " + t["b"]
    table = t.pivot(index="row", columns="ETF", values="cell")[TICKERS + ["pooled"]]
    table.columns = [f"{c} {SECTORS[c]}" for c in TICKERS] + ["Pooled"]
    return table.rename_axis(None)

for loss in ["MSE_log", "QLIKE"]:
    print(f"{loss}: ratio (p-value), DM")
    display(by_sector(loss))